In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
import catboost
from sklearn.metrics import classification_report, confusion_matrix

from src.data import load_flattened_data
from src.features import get_train_class
from src.evaluate import evaluate_cost
from src.modeling import build_train_table, train_catboost, build_eval_table

In [2]:
# def make_snapshots(df: pd.DataFrame, fracs):
#     rows = []
#     n_len = df["vehicle_id"].nunique()
#     print(n_len)
#     n = 0
#     for vid, g in df.sort_values("time_step").groupby("vehicle_id"):
#         n += 1
#         if n % 500 == 0:
#             print(n)
#         t = g["time_step"].to_numpy()
#         for f in fracs:
#             cut = np.percentile(t, f * 100)
#             rows.append(g[g["time_step"] <= cut].iloc[-1])
#     snap = pd.DataFrame(rows).drop_duplicates(["vehicle_id", "time_step"])
#     snap["tte"] = (snap["length_of_study_time_step"] - snap["time_step"]).clip(lower=1e-3)
#     return snap.reset_index(drop=True)
# def predict_classes(model, X):
#     predict = model.predict_survival_function(X)
#     S = np.array([[s(t) for t in EDGES] for s in predict])
#     P = np.c_[S[:, 3], S[:, 2]-S[:, 3], S[:, 1]-S[:, 2], S[:, 0]-S[:, 1], 1-S[:, 0]]
#     return np.argmin(P @ COST, axis=1)
# def true_class(tte, event):
#     cls = 4 - np.searchsorted(EDGES, tte, side="left")
#     return np.where(~event & (tte <= EDGES[-1]), -1, cls)


In [3]:
df = load_flattened_data("train")
df = get_train_class(df, return_df=True)
df.head()

,vehicle_id,time_step,171_0,666_0,427_0,837_0,167_0,167_1,167_2,167_3,...,Spec_2,Spec_3,Spec_4,Spec_5,Spec_6,Spec_7,length_of_study_time_step,in_study_repair,tte,class_label
0,0,11.2,167985.0,10787.0,7413813.0,2296.0,4110.0,1296420.0,1628265.0,630345.0,...,Cat0,Cat0,Cat0,Cat0,Cat0,Cat0,510.0,0,498.8,0
1,0,11.4,167985.0,10787.0,7413813.0,2296.0,4111.0,1302855.0,1628265.0,630345.0,...,Cat0,Cat0,Cat0,Cat0,Cat0,Cat0,510.0,0,498.6,0
2,0,19.6,331635.0,14525.0,13683604.0,2600.0,NaN,NaN,NaN,NaN,...,Cat0,Cat0,Cat0,Cat0,Cat0,Cat0,510.0,0,490.4,0
3,0,20.2,354975.0,15015.0,14540449.0,2616.0,NaN,NaN,NaN,NaN,...,Cat0,Cat0,Cat0,Cat0,Cat0,Cat0,510.0,0,489.8,0
4,0,21.0,365550.0,15295.0,14966985.0,2720.0,NaN,NaN,NaN,NaN,...,Cat0,Cat0,Cat0,Cat0,Cat0,Cat0,510.0,0,489.0,0


In [10]:
X, y, groups, weights = build_train_table(df)

In [5]:
model = train_catboost(X, y, n_estimators=100)

Learning rate set to 0.5
0:	learn: 0.3724814	total: 1.35s	remaining: 2m 13s
1:	learn: 0.2594708	total: 3.29s	remaining: 2m 41s
2:	learn: 0.2044189	total: 4.97s	remaining: 2m 40s
3:	learn: 0.1759728	total: 6.19s	remaining: 2m 28s
4:	learn: 0.1602582	total: 7.6s	remaining: 2m 24s
5:	learn: 0.1518279	total: 8.96s	remaining: 2m 20s
6:	learn: 0.1470398	total: 10.3s	remaining: 2m 16s
7:	learn: 0.1438099	total: 11.6s	remaining: 2m 13s
8:	learn: 0.1413933	total: 12.9s	remaining: 2m 10s
9:	learn: 0.1401909	total: 14s	remaining: 2m 6s
10:	learn: 0.1389885	total: 15.3s	remaining: 2m 3s
11:	learn: 0.1383266	total: 16.9s	remaining: 2m 4s
12:	learn: 0.1375647	total: 18.3s	remaining: 2m 2s
13:	learn: 0.1366917	total: 19.7s	remaining: 2m 1s
14:	learn: 0.1359216	total: 21.2s	remaining: 2m
15:	learn: 0.1349823	total: 22.8s	remaining: 1m 59s
16:	learn: 0.1343972	total: 24.5s	remaining: 1m 59s
17:	learn: 0.1339416	total: 26.5s	remaining: 2m
18:	learn: 0.1332875	total: 28.2s	remaining: 2m
19:	learn: 0.1329

In [6]:
vehicle_last_idx = groups.groupby(groups).tail(1).index
y_pred = model.predict(X.loc[vehicle_last_idx])
y_true = y[vehicle_last_idx]
print("All class 0 cost:", evaluate_cost(y_true, np.full(y_true.shape, 0))[0])
print("All class 4 cost:", evaluate_cost(y_true, np.full(y_true.shape, 4))[0])
print("Predict class cost:", evaluate_cost(y_true, y_pred[:, 0])[0])
print(classification_report(y_true, y_pred))
print(confusion_matrix(y_true, y_pred))

All class 0 cost: 1091600
All class 4 cost: 214836
Predict class cost: 1038673
              precision    recall  f1-score   support

           0       0.91      1.00      0.95     21287
           1       0.00      0.00      0.00        31
           2       0.02      0.01      0.02        70
           3       0.00      0.00      0.00       161
           4       0.95      0.03      0.06      2000

    accuracy                           0.91     23549
   macro avg       0.38      0.21      0.21     23549
weighted avg       0.90      0.91      0.87     23549

[[21278     5     1     0     3]
 [   31     0     0     0     0]
 [   68     1     1     0     0]
 [  156     1     4     0     0]
 [ 1825    45    44    23    63]]


In [7]:
df_val = load_flattened_data("validation")

In [11]:
x_val, y_val, groups_val = build_eval_table(df_val)

In [9]:
y_pred = model.predict(x_val)
print("All class 0 cost:", evaluate_cost(y_val, np.full(y_val.shape, 0))[0])
print("All class 4 cost:", evaluate_cost(y_val, np.full(y_val.shape, 4))[0])
print("Predict class cost:", evaluate_cost(y_val, y_pred[:, 0])[0])
print(classification_report(y_val, y_pred, zero_division=0))
print(confusion_matrix(y_val, y_pred))

All class 0 cost: 57400
All class 4 cost: 49566
Predict class cost: 57089
              precision    recall  f1-score   support

           0       0.97      0.92      0.94      4910
           1       0.00      0.00      0.00        16
           2       0.00      0.00      0.00        14
           3       0.00      0.00      0.00        30
           4       0.03      0.04      0.03        76

    accuracy                           0.89      5046
   macro avg       0.20      0.19      0.20      5046
weighted avg       0.95      0.89      0.92      5046

[[4505  122   56  115  112]
 [  14    0    0    0    2]
 [  13    0    0    1    0]
 [  27    0    2    0    1]
 [  68    3    1    1    3]]
